# Exploratory factor analysis

Principal axis factoring with oblimin rotation, as in step 4, plus the cross-check against the independent `factor_analyzer` package.

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "../analysis")
import psychometrics as ps

DATA = Path("../web/public/data")
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def load(name):
    df = pd.read_csv(DATA / f"{name}.csv")
    meta = json.loads((DATA / f"{name}.json").read_text())
    return df, meta


In [2]:
df, meta = load("writing_ai_simulated")
X = ps.reverse(ps.listwise(df.to_numpy(float)), [df.columns.get_loc("CE4")], *meta["scale"])
res = ps.efa(ps.corr(X), 3, "oblimin")
L = pd.DataFrame(res["loadings"], index=df.columns, columns=["F1", "F2", "F3"])
L["h2"] = res["communalities"]
print("RMSR:", round(res["rmsr"], 3))
L.style.format("{:.2f}").map(lambda v: "font-weight: bold" if abs(v) >= .40 else "", subset=["F1", "F2", "F3"])

RMSR: 0.022


,F1,F2,F3,h2
SE1,0.73,0.02,0.02,0.53
SE2,0.69,-0.01,0.03,0.49
SE3,0.70,0.04,-0.01,0.47
SE4,0.73,0.06,-0.00,0.51
SE5,0.36,-0.01,0.02,0.14
CE1,0.04,-0.07,0.72,0.55
CE2,-0.02,0.01,0.63,0.39
CE3,0.02,0.05,0.66,0.44
CE4,-0.05,0.06,0.66,0.41
CE5,0.02,-0.02,0.60,0.37


In [3]:
pd.DataFrame(res["phi"], index=["F1", "F2", "F3"], columns=["F1", "F2", "F3"]).round(2)

,F1,F2,F3
F1,1.00,-0.31,0.26
F2,-0.31,1.00,-0.16
F3,0.26,-0.16,1.00


In [4]:
from factor_analyzer import FactorAnalyzer
fa = FactorAnalyzer(n_factors=3, rotation="oblimin", method="minres").fit(X)
ours, theirs = res["loadings"], fa.loadings_
C = (ours.T @ theirs) / np.sqrt(np.outer((ours**2).sum(0), (theirs**2).sum(0)))
print("Tucker congruence with factor_analyzer (minres), per factor:", np.round(np.abs(C).max(axis=1), 4))

Tucker congruence with factor_analyzer (minres), per factor: [1. 1. 1.]
